# 문서 분류 (Document Classification)

# 1 나이브 베이즈 분류(Naive Bayes Classifier)

## 1.1 직접구현

### Naive Bayes Classifier

In [1]:
training_set = [
    ['me free lottery', 1],
    ['free get free you', 1],
    ['you free scholarship', 0],
    ['free to contact me', 0],
    ['you won award', 0],
    ['you ticket lottery', 1]
]

### 토큰 빈도수 및 문서별 토큰수 계산 (확률 계산을 위한 준비)

![대체 텍스트](https://wikimedia.org/api/rest_v1/media/math/render/svg/98f086c560aa2f66650060277dda4f90e54e30c0)

In [ ]:
unique_tokens = set()
for doc, y in training_set:
    unique_tokens.update(doc.split())

unique_tokens
word2idx = {t: i for i, t in enumerate(unique_tokens)}
word2idx

{'won': 0,
 'me': 1,
 'to': 2,
 'ticket': 3,
 'get': 4,
 'scholarship': 5,
 'award': 6,
 'contact': 7,
 'lottery': 8,
 'you': 9,
 'free': 10}

In [11]:
import pandas as pd
# temp = [
#     [0, 0],
# ]
# df = pd.DataFrame(temp, index=['won'])
# display('df1', df)
len_y = len( set(y for _, y in training_set) ) # 0, 1 -> 2개

temp = {t: [0]*len_y for t in unique_tokens}
# {'won': [0, 0], }

### Training : 토큰별 조건부 확률 계산

In [ ]:
for doc, y in training_set:
    for token in doc.split():
        temp[token][y] += 1

In [ ]:
target_tokens = [ [] for _ in range(len_y) ]
print(target_tokens)
for doc, y in training_set:
    target_tokens[y] += doc.split()
print(*target_tokens, sep='\n')

from collections import Counter
token_counts = []
for tokens in target_tokens:
    token_counts.append(Counter(tokens))
pd.DataFrame(token_counts).fillna(0).astype(int).T

[[], []]
['you', 'free', 'scholarship', 'free', 'to', 'contact', 'me', 'you', 'won', 'award']
['me', 'free', 'lottery', 'free', 'get', 'free', 'you', 'you', 'ticket', 'lottery']


,0,1
you,2,2
free,2,3
scholarship,1,0
to,1,0
contact,1,0
me,1,1
won,1,0
award,1,0
lottery,0,2
get,0,1


In [26]:
df = pd.DataFrame(temp, index=['normal', 'spam']).T

targets = df.columns

k = 0.5
for target in targets:
    column_name = f'P(w|{target})'
    df[column_name] = (k + df[target]) / (2 * k + df[target].sum())

In [ ]:
import numpy as np
for target in targets:
    conditional_column = f'P(w|{target})'
    log_column = f'Log(P(w|{target}))'
    df[log_column] = np.log(df[conditional_column])
df

,normal,spam,P(w|normal),P(w|spam),Log(P(w|normal)),Log(P(w|spam))
won,1,0,0.136364,0.045455,-1.992430,-3.091042
me,1,1,0.136364,0.136364,-1.992430,-1.992430
to,1,0,0.136364,0.045455,-1.992430,-3.091042
ticket,0,1,0.045455,0.136364,-3.091042,-1.992430
get,0,1,0.045455,0.136364,-3.091042,-1.992430
scholarship,1,0,0.136364,0.045455,-1.992430,-3.091042
award,1,0,0.136364,0.045455,-1.992430,-3.091042
contact,1,0,0.136364,0.045455,-1.992430,-3.091042
lottery,0,2,0.045455,0.227273,-3.091042,-1.481605
you,2,2,0.227273,0.227273,-1.481605,-1.481605


In [35]:
# Log(P(normal)), Log(P(spam))
log_p_targets = np.log(df[targets].sum() / df[targets].sum().sum()).to_dict()
log_p_targets

{'normal': -0.6931471805599453, 'spam': -0.6931471805599453}

### Classify : 신규 텍스트가 주어졌을 때 확률 계산

In [ ]:
doc = 'free lottery none'

result = {}
for target in targets:
    temp = log_p_targets[target]
    for token in doc.split():
        log_column = f'Log(P(w|{target}))'
        temp += df[log_column].get(token, 0.0) # np.log(k + 2*k)
    result[target] = np.exp(temp)

# result = {
#     'normal': 값1 <- log(p(normal)) + log(p(free|normal)) + log(p(lottery|normal))
#     'spam': 값2 <- log(p(spam)) + log(p(free|spam)) + log(p(lottery|spam))
# }

print(result)

# 비율 계산
result = {k: v / sum(result.values()) for k, v in result.items()}

# result = {
#     'normal': 값1 / (값1+값2)
#     'spam': 값2 / (값1+값2)
# }

result

{'normal': 0.00516528925619835, 'spam': 0.03615702479338842}


{'normal': 0.12500000000000008, 'spam': 0.8749999999999999}

## 1.2 sklearn 활용 (영문 뉴스 분류)

- naive_bayes.MultinomialNB() : 빈도수 기반 Naive Bayse Classifier

### 뉴스 데이터 다운로드



In [47]:
from sklearn.datasets import fetch_20newsgroups
twenty_train = fetch_20newsgroups(subset='train', shuffle=True)
print(twenty_train.target_names) #뉴스 카테고리 출력
print(twenty_train.data[0]) #뉴스 데이터 출력

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have o

### 문서 분류(파이프 라인 사용)

In [48]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB

text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_clf = text_clf.fit(twenty_train.data, twenty_train.target)

In [ ]:
CountVectorizer

In [49]:
import numpy as np
twenty_test = fetch_20newsgroups(subset='test', shuffle=True)
predicted = text_clf.predict(twenty_test.data)
np.mean(predicted == twenty_test.target)

0.7738980350504514

### Grid Search

In [50]:
from sklearn.model_selection import GridSearchCV
parameters_clf = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'tfidf__use_idf': (True, False),
}
gs_clf = GridSearchCV(text_clf, parameters_clf, n_jobs=-1, verbose=2)
gs_clf = gs_clf.fit(twenty_train.data, twenty_train.target)
print("Best score: {0}".format(gs_clf.best_score_))
print("Best parameters set:")
best_parameters = gs_clf.best_estimator_.get_params()
for param_name in sorted(list(best_parameters.keys())):
    print("\t{0}: {1}".format(param_name, best_parameters[param_name]))

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best score: 0.8518650274101537
Best parameters set:
	clf: MultinomialNB()
	clf__alpha: 1.0
	clf__class_prior: None
	clf__fit_prior: True
	clf__force_alpha: True
	memory: None
	steps: [('vect', CountVectorizer(ngram_range=(1, 2))), ('tfidf', TfidfTransformer()), ('clf', MultinomialNB())]
	tfidf: TfidfTransformer()
	tfidf__norm: l2
	tfidf__smooth_idf: True
	tfidf__sublinear_tf: False
	tfidf__use_idf: True
	transform_input: None
	vect: CountVectorizer(ngram_range=(1, 2))
	vect__analyzer: word
	vect__binary: False
	vect__decode_error: strict
	vect__dtype: <class 'numpy.int64'>
	vect__encoding: utf-8
	vect__input: content
	vect__lowercase: True
	vect__max_df: 1.0
	vect__max_features: None
	vect__min_df: 1
	vect__ngram_range: (1, 2)
	vect__preprocessor: None
	vect__stop_words: None
	vect__strip_accents: None
	vect__token_pattern: (?u)\b\w\w+\b
	vect__tokenizer: None
	vect__vocabulary: None
	verbose: False


### Parameter 적용

In [51]:
import numpy as np
predicted = gs_clf.best_estimator_.predict(twenty_test.data)
np.mean(predicted == twenty_test.target)

0.765400955921402

# 2 서포트 벡터 머신(SVM, Support Vector Machine)

- linear_model.SGDClassifier() : 선형 경사하강법 분류 모델

### 뉴스 데이터 다운로드

In [52]:
from sklearn.datasets import fetch_20newsgroups
twenty_train = fetch_20newsgroups(subset='train', shuffle=True)
print(twenty_train.target_names) #뉴스 카테고리 출력
print(twenty_train.data[0]) #뉴데 이터 출력

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have o

### 문서 분류 (파이프 라인 사용)

In [53]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
text_clf_svm = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf-svm', SGDClassifier(loss='hinge', penalty='l2', alpha=1e-3, n_iter_no_change=5, random_state=42)),
])
text_clf_svm = text_clf_svm.fit(twenty_train.data, twenty_train.target)
predicted_svm = text_clf_svm.predict(twenty_test.data)
np.mean(predicted_svm == twenty_test.target)

0.8240839086563994

### Grid Search

In [54]:
from sklearn.model_selection import GridSearchCV
parameters_svm = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'tfidf__use_idf': (True, False),
    'clf-svm__alpha': (1e-2, 1e-3),
}
gs_clf_svm = GridSearchCV(text_clf_svm, parameters_svm, n_jobs=-1)
gs_clf_svm = gs_clf_svm.fit(twenty_train.data, twenty_train.target)
gs_clf_svm.best_score_
gs_clf_svm.best_params_

{'clf-svm__alpha': 0.001, 'tfidf__use_idf': True, 'vect__ngram_range': (1, 2)}

In [55]:
print("Best score: {0}".format(gs_clf_svm.best_score_))
print("Best parameters set:")
best_parameters = gs_clf_svm.best_estimator_.get_params()
for param_name in sorted(list(best_parameters.keys())):
    print("\t{0}: {1}".format(param_name, best_parameters[param_name]))

Best score: 0.9051618841994754
Best parameters set:
	clf-svm: SGDClassifier(alpha=0.001, random_state=42)
	clf-svm__alpha: 0.001
	clf-svm__average: False
	clf-svm__class_weight: None
	clf-svm__early_stopping: False
	clf-svm__epsilon: 0.1
	clf-svm__eta0: 0.0
	clf-svm__fit_intercept: True
	clf-svm__l1_ratio: 0.15
	clf-svm__learning_rate: optimal
	clf-svm__loss: hinge
	clf-svm__max_iter: 1000
	clf-svm__n_iter_no_change: 5
	clf-svm__n_jobs: None
	clf-svm__penalty: l2
	clf-svm__power_t: 0.5
	clf-svm__random_state: 42
	clf-svm__shuffle: True
	clf-svm__tol: 0.001
	clf-svm__validation_fraction: 0.1
	clf-svm__verbose: 0
	clf-svm__warm_start: False
	memory: None
	steps: [('vect', CountVectorizer(ngram_range=(1, 2))), ('tfidf', TfidfTransformer()), ('clf-svm', SGDClassifier(alpha=0.001, random_state=42))]
	tfidf: TfidfTransformer()
	tfidf__norm: l2
	tfidf__smooth_idf: True
	tfidf__sublinear_tf: False
	tfidf__use_idf: True
	transform_input: None
	vect: CountVectorizer(ngram_range=(1, 2))
	vect__an

In [56]:
import numpy as np
predicted = gs_clf_svm.best_estimator_.predict(twenty_test.data)
np.mean(predicted == twenty_test.target)

0.8351035581518853